# Task 4 - Specialist vs Generalist Analysis

This notebook is the comparison notebook for the paper. It assumes the full run from
Task 2 has produced saved adapters and per-scenario results, then adds the missing
analysis for generalist vs planner-plus-specialists.

The key repo-specific bug here is that the current pipeline evaluation loses dependency
information and rewrites every filled specialist step as `Dependency: None`. That makes
`ArgKey` and `ArgVal` look weaker than they should be.


## Repo Understanding

This project is a forked AssetOpsBench workflow for training Gemma to plan without tool descriptions.

- `src/servers/` contains the local MCP servers: `iot`, `fmsr`, `tsfm`, and `utilities`.
- `src/workflow/` contains the plan / execute runner used by the benchmark scripts.
- `benchmark/generate_data/datasets/` contains the three SFT datasets used by the current notebook:
  `tool_knowledge.jsonl`, `planning.jsonl`, and `execution.jsonl`.
- `benchmark/baseline_tests/gemini_flash_informed_results.json` is the main gold-plan file used by evaluation.
- `notebook/AssetOpsBench_Gemma_FineTuning.ipynb` is the current end-to-end Colab notebook.

Current notebook gaps that matter for the TODO list:

- training still uses `report_to="none"` in `SFTConfig`
- model loading still uses `torch_dtype=...` instead of `dtype=...`
- pipeline evaluation drops dependencies by hardcoding `Dependency: None`
- the specialist pipeline currently logs no latency analysis, confusion matrix, or W&B tables


In [ ]:
!pip install -U transformers peft trl accelerate bitsandbytes
!pip install -U datasets evaluate rouge-score pandas matplotlib seaborn scikit-learn
!pip install -U wandb sentencepiece protobuf


In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import warnings
import inspect
from copy import deepcopy
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import wandb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu:", torch.cuda.get_device_name(0), f"({props.total_memory / 1e9:.1f} GB)")


In [ ]:
WANDB_PROJECT = "hpml-group20-assetops"
LIGHT_MODE = True
MODEL_ID = "google/gemma-4-E4B-it"
LOAD_IN_8BIT = True
LOAD_IN_4BIT = False
HF_TOKEN = os.environ.get("HF_TOKEN", "")

REPO_URL = "https://github.com/YuvalShemla/hpml-2026-project.git"
REPO_DIR = Path("/content/hpml-2026-project")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 16 if LIGHT_MODE else 32
LORA_ALPHA = 32 if LIGHT_MODE else 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 40
if GPU_MEM_GB > 60:
    MAX_SEQ_LENGTH = 1024 if LIGHT_MODE else 2048
    PER_DEVICE_BATCH_SIZE = 2 if LIGHT_MODE else 4
    GRADIENT_ACCUMULATION_STEPS = 4
else:
    MAX_SEQ_LENGTH = 512 if LIGHT_MODE else 1024
    PER_DEVICE_BATCH_SIZE = 1 if LIGHT_MODE else 2
    GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 2e-4
NUM_EPOCHS = 1 if LIGHT_MODE else 3
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
BF16 = True
NUM_HELD_OUT = 15 if LIGHT_MODE else 50
MAX_NEW_TOKENS = 512 if LIGHT_MODE else 1024
TEMPERATURE = 0.1
TOP_P = 0.9
MAX_TRAIN_EXAMPLES = 300 if LIGHT_MODE else None

RUN_TAGS = ["light-mode"] if LIGHT_MODE else ["full-run"]
RUN_GROUP = "task-notebook"
print(
    {
        "light_mode": LIGHT_MODE,
        "model": MODEL_ID,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lr": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "held_out": NUM_HELD_OUT,
    }
)


In [ ]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo already present:", REPO_DIR)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

datasets_dir = REPO_DIR / "benchmark" / "generate_data" / "datasets"
ds_tool = load_jsonl(datasets_dir / "tool_knowledge.jsonl")
ds_plan = load_jsonl(datasets_dir / "planning.jsonl")
ds_exec = load_jsonl(datasets_dir / "execution.jsonl")

gold_path = REPO_DIR / "benchmark" / "baseline_tests" / "gemini_flash_informed_results.json"
gold_plans_raw = load_json(gold_path)
gold_plans = {row["id"]: row for row in gold_plans_raw if row.get("plan_steps", 0) > 0}

print(
    {
        "tool_examples": len(ds_tool),
        "planning_examples": len(ds_plan),
        "execution_examples": len(ds_exec),
        "gold_plans": len(gold_plans),
    }
)


In [ ]:
from datasets import load_dataset

hf_ds = load_dataset("ibm-research/AssetOpsBench", "scenarios")
hf_scenarios = [dict(row) for row in hf_ds["train"]]

eval_candidates = []
for sc in hf_scenarios:
    if sc["id"] in gold_plans:
        eval_candidates.append(
            {
                "id": sc["id"],
                "question": sc["text"],
                "type": sc["type"],
                "category": sc["category"],
                "gold_plan": gold_plans[sc["id"]]["response"],
                "gold_steps": gold_plans[sc["id"]]["plan_steps"],
                "gold_agents": gold_plans[sc["id"]].get("agents_used", []),
                "gold_tools": gold_plans[sc["id"]].get("tools_used", []),
            }
        )

random.shuffle(eval_candidates)
by_type = defaultdict(list)
for sc in eval_candidates:
    by_type[sc["type"]].append(sc)

eval_scenarios = []
per_type_quota = max(1, NUM_HELD_OUT // len(by_type))
for scenario_type, rows in by_type.items():
    eval_scenarios.extend(rows[: min(per_type_quota, len(rows))])

remaining = NUM_HELD_OUT - len(eval_scenarios)
eval_ids = {row["id"] for row in eval_scenarios}
for scenario_type in sorted(by_type, key=lambda key: -len(by_type[key])):
    for row in by_type[scenario_type]:
        if remaining <= 0:
            break
        if row["id"] not in eval_ids:
            eval_scenarios.append(row)
            eval_ids.add(row["id"])
            remaining -= 1

eval_questions_lower = {row["question"].strip().lower() for row in eval_scenarios}
held_out_plans = {row["gold_plan"].strip() for row in eval_scenarios}

def is_leaked(example):
    return example["messages"][0]["content"].strip().lower() in eval_questions_lower

clean_tool = [row for row in ds_tool if not is_leaked(row)]
clean_plan = [row for row in ds_plan if not is_leaked(row)]
clean_exec = [row for row in ds_exec if not is_leaked(row)]
clean_plan = [row for row in clean_plan if row["messages"][1]["content"].strip() not in held_out_plans]

print("held_out:", len(eval_scenarios))
print("type_distribution:", dict(Counter(row["type"] for row in eval_scenarios)))
print("clean_train_total:", len(clean_tool) + len(clean_plan) + len(clean_exec))


In [ ]:
from evaluate import load as load_metric

rouge_metric = load_metric("rouge")

VALID_AGENTS = {
    "IoTAgent",
    "FMSRAgent",
    "TSFMAgent",
    "Utilities",
    "WorkOrderAgent",
    "VibrationAgent",
    "none",
}

VALID_TOOLS = {
    "sites",
    "assets",
    "sensors",
    "history",
    "get_failure_modes",
    "get_failure_mode_sensor_mapping",
    "get_ai_tasks",
    "get_tsfm_models",
    "run_tsfm_forecasting",
    "run_tsfm_finetuning",
    "run_tsad",
    "run_integrated_tsad",
    "json_reader",
    "current_date_time",
    "current_time_english",
    "get_work_orders",
    "get_preventive_work_orders",
    "get_corrective_work_orders",
    "get_events",
    "get_failure_codes",
    "get_work_order_distribution",
    "predict_next_work_order",
    "analyze_alert_to_failure",
    "get_vibration_data",
    "list_vibration_sensors",
    "compute_fft_spectrum",
    "compute_envelope_spectrum",
    "assess_vibration_severity",
    "calculate_bearing_frequencies",
    "list_known_bearings",
    "diagnose_vibration",
    "none",
}

TOOL_DESCRIPTIONS = """Available Agents and Tools:

IoTAgent:
  - sites()
  - assets(site_name)
  - sensors(site_name, asset_id)
  - history(site_name, asset_id, start, final=None)

FMSRAgent:
  - get_failure_modes(asset_name)
  - get_failure_mode_sensor_mapping(asset_name, failure_modes, sensors)

TSFMAgent:
  - get_ai_tasks()
  - get_tsfm_models()
  - run_tsfm_forecasting(...)
  - run_tsfm_finetuning(...)
  - run_tsad(...)
  - run_integrated_tsad(...)

Utilities:
  - json_reader(file_name)
  - current_date_time()
  - current_time_english()

WorkOrderAgent:
  - get_work_orders(asset_name)
  - get_preventive_work_orders(asset_name)
  - get_corrective_work_orders(asset_name)
  - get_events(asset_name)
  - get_failure_codes(asset_name)
  - get_work_order_distribution(asset_name)
  - predict_next_work_order(asset_name)
  - analyze_alert_to_failure(asset_name)
"""

INFORMED_PROMPT = """You are an expert planner for industrial asset operations. Given a question and available tools, produce a structured plan.

{tool_descriptions}

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Use only listed agents/tools. Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""

BLIND_PROMPT = """You are an expert planner for industrial asset operations. Produce a structured plan.

OUTPUT FORMAT:
#Task1: <description>
#Agent1: <agent_name>
#Tool1: <tool_name>
#Args1: {{"arg": "value"}}
#Dependency1: None
#ExpectedOutput1: <what to expect>

Rules: Keep plans concise. Use {{step_N}} for dependencies.

QUESTION: {question}
"""

def parse_plan(text):
    steps = []
    if not text:
        return steps
    for block in re.split(r"(?=#Task\d+:)", text):
        block = block.strip()
        task_m = re.search(r"#Task(\d+):\s*(.*)", block)
        agent_m = re.search(r"#Agent\d+:\s*(\S+)", block)
        tool_m = re.search(r"#Tool\d+:\s*(\S+)", block)
        args_m = re.search(r"#Args\d+:\s*(.*)", block)
        dep_m = re.search(r"#Dependency\d+:\s*(.*)", block)
        exp_m = re.search(r"#ExpectedOutput\d+:\s*(.*)", block)
        if not task_m:
            continue
        args_raw = args_m.group(1).strip() if args_m else "{}"
        try:
            args = json.loads(args_raw)
        except Exception:
            args = {}
        steps.append(
            {
                "step": int(task_m.group(1)),
                "task": task_m.group(2).strip(),
                "agent": agent_m.group(1).strip() if agent_m else "",
                "tool": tool_m.group(1).strip().rstrip("()") if tool_m else "",
                "args": args,
                "args_raw": args_raw,
                "dependency": dep_m.group(1).strip() if dep_m else "None",
                "expected_output": exp_m.group(1).strip() if exp_m else "",
            }
        )
    return steps

def compute_set_f1(gold_items, pred_items):
    gold_set = set(gold_items)
    pred_set = set(pred_items)
    if not gold_set and not pred_set:
        return 1.0
    if not gold_set or not pred_set:
        return 0.0
    tp = len(gold_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

def dependency_edge_set(steps):
    edges = set()
    for step in steps:
        dep_field = step.get("dependency", "")
        dep_ids = re.findall(r"(?:step_|#S)?(\d+)", dep_field)
        for dep_id in dep_ids:
            edges.add((int(dep_id), step["step"]))
    return edges

def evaluate_plan(pred_text, gold_text):
    pred_steps = parse_plan(pred_text)
    gold_steps = parse_plan(gold_text)

    pred_pairs = [(step["agent"], step["tool"]) for step in pred_steps if step["tool"]]
    gold_pairs = [(step["agent"], step["tool"]) for step in gold_steps if step["tool"]]

    gold_args = {(step["agent"], step["tool"]): step["args"] for step in gold_steps if step["args"]}

    key_scores = []
    value_scores = []
    for step in pred_steps:
        key = (step["agent"], step["tool"])
        if key not in gold_args:
            continue
        gold_arg_dict = gold_args[key]
        pred_arg_dict = step["args"] if isinstance(step["args"], dict) else {}
        key_scores.append(compute_set_f1(gold_arg_dict.keys(), pred_arg_dict.keys()))
        shared = set(gold_arg_dict) & set(pred_arg_dict)
        if shared:
            value_scores.append(
                np.mean(
                    [
                        1.0
                        if str(gold_arg_dict[k]).strip().lower() == str(pred_arg_dict[k]).strip().lower()
                        else 0.0
                        for k in shared
                    ]
                )
            )

    dep_f1 = compute_set_f1(dependency_edge_set(gold_steps), dependency_edge_set(pred_steps))
    rouge = rouge_metric.compute(predictions=[pred_text], references=[gold_text])["rougeL"]

    return {
        "has_plan_format": "#Task" in pred_text and "#Agent" in pred_text and "#Tool" in pred_text,
        "num_steps": len(pred_steps),
        "gold_steps": len(gold_steps),
        "agent_tool_f1": compute_set_f1(gold_pairs, pred_pairs),
        "arg_key_f1": float(np.mean(key_scores)) if key_scores else 0.0,
        "arg_value_match": float(np.mean(value_scores)) if value_scores else 0.0,
        "dependency_f1": dep_f1,
        "rouge_l": rouge,
        "valid_agents": sum(1 for step in pred_steps if step["agent"] in VALID_AGENTS),
        "total_agents": len(pred_steps),
        "valid_tools": sum(1 for step in pred_steps if step["tool"] in VALID_TOOLS),
        "total_tools": len(pred_steps),
        "pred_steps": pred_steps,
        "gold_steps_parsed": gold_steps,
    }

def summarize_results(results, mode_name):
    if not results:
        return {"mode": mode_name, "total": 0}
    return {
        "mode": mode_name,
        "total": len(results),
        "format_valid_pct": 100 * np.mean([row["has_plan_format"] for row in results]),
        "avg_agent_tool_f1": float(np.mean([row["agent_tool_f1"] for row in results])),
        "avg_arg_key_f1": float(np.mean([row["arg_key_f1"] for row in results])),
        "avg_arg_value_match": float(np.mean([row["arg_value_match"] for row in results])),
        "avg_dependency_f1": float(np.mean([row["dependency_f1"] for row in results])),
        "avg_rouge_l": float(np.mean([row["rouge_l"] for row in results])),
        "agent_correctness": sum(row["valid_agents"] for row in results) / max(sum(row["total_agents"] for row in results), 1),
        "tool_correctness": sum(row["valid_tools"] for row in results) / max(sum(row["total_tools"] for row in results), 1),
    }

def print_summary(summary):
    print(json.dumps(summary, indent=2))


In [ ]:
def strip_plan_to_routing(plan_text):
    keep = []
    for line in plan_text.splitlines():
        stripped = line.strip()
        if not stripped:
            keep.append("")
            continue
        if stripped.startswith(("#Task", "#Agent", "#Tool", "#Dependency")):
            keep.append(line)
    return "\n".join(keep).strip()

def extract_specialist_steps(plan_text):
    steps = []
    for step in parse_plan(plan_text):
        if step["agent"] in ("", "none"):
            continue
        instruction = (
            "Generate the arguments for this tool call:\n"
            f"Task: {step['task']}\n"
            f"Agent: {step['agent']}\n"
            f"Tool: {step['tool']}\n"
            f"Dependency: {step['dependency']}"
        )
        response = f"#Args: {json.dumps(step['args'])}\n#ExpectedOutput: {step['expected_output'] or 'Result'}"
        steps.append({"agent": step["agent"], "instruction": instruction, "response": response})
    return steps

def build_training_sets():
    all_train = [{"messages": row["messages"]} for row in clean_tool + clean_plan + clean_exec]
    random.shuffle(all_train)
    if MAX_TRAIN_EXAMPLES:
        all_train = all_train[:MAX_TRAIN_EXAMPLES]
    split = int(len(all_train) * 0.95)
    generalist_train, generalist_eval = all_train[:split], all_train[split:]

    planner_data = []
    for row in clean_plan:
        if row.get("metadata", {}).get("category") != "planning":
            continue
        routing = strip_plan_to_routing(row["messages"][1]["content"])
        if "#Task" in routing and "#Agent" in routing:
            planner_data.append(
                {
                    "messages": [
                        {"role": "user", "content": row["messages"][0]["content"]},
                        {"role": "assistant", "content": routing},
                    ]
                }
            )
    for row in clean_tool:
        planner_data.append({"messages": row["messages"]})
    random.shuffle(planner_data)
    if MAX_TRAIN_EXAMPLES:
        planner_data = planner_data[:MAX_TRAIN_EXAMPLES]
    split = int(len(planner_data) * 0.95)
    planner_train, planner_eval = planner_data[:split], planner_data[split:]

    specialist_data = defaultdict(list)
    for row in clean_plan:
        if row.get("metadata", {}).get("category") != "planning":
            continue
        for step in extract_specialist_steps(row["messages"][1]["content"]):
            specialist_data[step["agent"]].append(
                {
                    "messages": [
                        {"role": "user", "content": step["instruction"]},
                        {"role": "assistant", "content": step["response"]},
                    ]
                }
            )

    return generalist_train, generalist_eval, planner_train, planner_eval, specialist_data


In [ ]:
RESULTS_ROOT = Path("/content/output_full_run")
GENERALIST_RESULTS_PATH = RESULTS_ROOT / "generalist_results.json"
PLANNER_RESULTS_PATH = RESULTS_ROOT / "planner_results.json"
PIPELINE_RESULTS_PATH = RESULTS_ROOT / "pipeline_results.json"

def safe_load_results(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    print("missing:", path)
    return []

generalist_results = safe_load_results(GENERALIST_RESULTS_PATH)
planner_results = safe_load_results(PLANNER_RESULTS_PATH)
pipeline_results = safe_load_results(PIPELINE_RESULTS_PATH)


In [ ]:
def scenario_bucket(scenario_type):
    scenario_type = scenario_type.lower()
    if "multi" in scenario_type:
        return "multiagent"
    return scenario_type

def attach_bucket(results):
    rows = []
    for row in results:
        new_row = dict(row)
        new_row["bucket"] = scenario_bucket(row.get("type", "unknown"))
        new_row["is_multiagent"] = new_row["bucket"] == "multiagent"
        rows.append(new_row)
    return rows

generalist_results = attach_bucket(generalist_results)
planner_results = attach_bucket(planner_results)
pipeline_results = attach_bucket(pipeline_results)


In [ ]:
gen_by_id = {row["id"]: row for row in generalist_results}
pipe_by_id = {row["id"]: row for row in pipeline_results}
plan_by_id = {row["id"]: row for row in planner_results}

comparison_rows = []
for scenario in eval_scenarios:
    sid = scenario["id"]
    gen = gen_by_id.get(sid, {})
    pipe = pipe_by_id.get(sid, {})
    planner = plan_by_id.get(sid, {})
    comparison_rows.append(
        {
            "id": sid,
            "type": scenario["type"],
            "bucket": scenario_bucket(scenario["type"]),
            "generalist_atf1": gen.get("agent_tool_f1", 0.0),
            "generalist_argkey": gen.get("arg_key_f1", 0.0),
            "generalist_argval": gen.get("arg_value_match", 0.0),
            "pipeline_atf1": pipe.get("agent_tool_f1", 0.0),
            "pipeline_argkey": pipe.get("arg_key_f1", 0.0),
            "pipeline_argval": pipe.get("arg_value_match", 0.0),
            "planner_atf1": planner.get("agent_tool_f1", 0.0),
            "generalist_win": gen.get("agent_tool_f1", 0.0) > pipe.get("agent_tool_f1", 0.0),
            "pipeline_win": pipe.get("agent_tool_f1", 0.0) > gen.get("agent_tool_f1", 0.0),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head()


In [ ]:
by_type = (
    comparison_df.groupby("bucket")
    .agg(
        count=("id", "count"),
        generalist_atf1=("generalist_atf1", "mean"),
        pipeline_atf1=("pipeline_atf1", "mean"),
        planner_atf1=("planner_atf1", "mean"),
        generalist_argkey=("generalist_argkey", "mean"),
        pipeline_argkey=("pipeline_argkey", "mean"),
        generalist_argval=("generalist_argval", "mean"),
        pipeline_argval=("pipeline_argval", "mean"),
    )
    .round(3)
)
display(by_type)

single_agent_df = comparison_df[comparison_df["bucket"] != "multiagent"]
multi_agent_df = comparison_df[comparison_df["bucket"] == "multiagent"]

print("single_agent_mean")
display(single_agent_df[["generalist_atf1", "pipeline_atf1", "planner_atf1"]].mean().to_frame("mean").T)
print("multi_agent_mean")
display(multi_agent_df[["generalist_atf1", "pipeline_atf1", "planner_atf1"]].mean().to_frame("mean").T)


In [ ]:
plt.figure(figsize=(10, 5))
chart_df = by_type.reset_index().melt(
    id_vars="bucket",
    value_vars=["generalist_atf1", "pipeline_atf1", "planner_atf1"],
    var_name="approach",
    value_name="at_f1",
)
sns.barplot(data=chart_df, x="bucket", y="at_f1", hue="approach")
plt.title("Generalist vs planner+specialists by scenario type")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Parallel-specialist latency estimate.
# Replace the constants below with measured times from actual inference logs once available.
planner_seconds = 1.0
specialist_seconds = {
    "IoTAgent": 0.6,
    "FMSRAgent": 0.6,
    "TSFMAgent": 0.8,
    "WorkOrderAgent": 0.7,
}
generalist_seconds = 2.2

latency_rows = []
for row in planner_results:
    agents = [step["agent"] for step in row.get("pred_steps", []) if step["agent"] not in ("", "none")]
    routed_agents = [agent for agent in agents if agent in specialist_seconds]
    serial_specialist = sum(specialist_seconds[agent] for agent in routed_agents)
    parallel_specialist = max([specialist_seconds[agent] for agent in routed_agents], default=0.0)
    latency_rows.append(
        {
            "id": row["id"],
            "routed_steps": len(routed_agents),
            "generalist_latency_s": generalist_seconds,
            "planner_plus_serial_specialists_s": planner_seconds + serial_specialist,
            "planner_plus_parallel_specialists_s": planner_seconds + parallel_specialist,
        }
    )

latency_df = pd.DataFrame(latency_rows)
latency_df.describe()


In [ ]:
failure_cases = comparison_df.sort_values(
    ["pipeline_atf1", "generalist_atf1"], ascending=[True, False]
).head(10)
print("generalist_beats_pipeline")
display(failure_cases[["id", "type", "generalist_atf1", "pipeline_atf1"]])

reverse_cases = comparison_df.sort_values(
    ["generalist_atf1", "pipeline_atf1"], ascending=[True, False]
).head(10)
print("pipeline_beats_generalist")
display(reverse_cases[["id", "type", "generalist_atf1", "pipeline_atf1"]])


In [ ]:
def perturb_planner_route(plan_text):
    steps = parse_plan(plan_text)
    if not steps:
        return plan_text
    candidate_agents = ["IoTAgent", "FMSRAgent", "TSFMAgent", "WorkOrderAgent"]
    first = steps[0]
    wrong_agents = [agent for agent in candidate_agents if agent != first["agent"]]
    if not wrong_agents:
        return plan_text
    wrong_agent = random.choice(wrong_agents)
    lines = plan_text.splitlines()
    new_lines = []
    for line in lines:
        if line.startswith(f"#Agent{first['step']}:"):
            new_lines.append(f"#Agent{first['step']}: {wrong_agent}")
        else:
            new_lines.append(line)
    return "\n".join(new_lines)

planner_error_rows = []
for row in planner_results[: min(20, len(planner_results))]:
    perturbed = perturb_planner_route(row["generated"])
    metrics = evaluate_plan(perturbed, next(sc["gold_plan"] for sc in eval_scenarios if sc["id"] == row["id"]))
    planner_error_rows.append(
        {
            "id": row["id"],
            "original_atf1": row.get("agent_tool_f1", 0.0),
            "perturbed_atf1": metrics["agent_tool_f1"],
            "original_dependency_f1": row.get("dependency_f1", 0.0),
            "perturbed_dependency_f1": metrics["dependency_f1"],
        }
    )
pd.DataFrame(planner_error_rows)
